# node_bootstrap.ipynb — livekernel 节点自举笔记本 (detect → reason → execute)

> 交付载具 v2: 用本笔记本 (几十 KB) 代替 48MB 整包上传。重依赖 (pip 包 / 代码)
> 由部署机器经镜像源自行获取; 笔记本只做"检测 → 推理 → 分阶段执行"。
> 零凭证 / 零 IP 字面量 / 零个人路径 — 可随仓库公开 (GitHub open-source ready)。

## 三模式

| 模式 | 环境变量 | 行为 |
|---|---|---|
| `detect` (默认) | `UEA_BOOT_MODE=detect` | 只读全域扫描 + 缺口报告, 不碰任何状态 |
| `execute` | `UEA_BOOT_MODE=execute` | 按 `UEA_BOOT_STAGES` 分阶段执行修复 (幂等) |
| `execute` + 冒烟 | 再加 `UEA_BOOT_SMOKE=1` | 执行后跑推理/引擎/黄金链冒烟 |

## 两载体

| 载体 | 环境变量 | 场景 |
|---|---|---|
| `local` (默认) | `UEA_BOOT_TRANSPORT=local` | 笔记本在节点上直跑 (jupyter / nbconvert) |
| `ssh` | `UEA_BOOT_TRANSPORT=ssh` + `UEA_NODE_HOST/USER/PWD` | 本地经 paramiko 驱动节点 (只读 detect 安全) |

## 阶段 (对齐 docs/PRD-P0-NODE-ALIGNMENT.md)

- **阶段 0** 安全止血: 旧副本 (带 credentials.json) 整目录替换 (D3), 先本地/节点备份再删
- **阶段 1** OpenClaw 接线: 本地 Omni-30B 设 primary, StepFun 留 fallback (D2), 清临时垫片
- **阶段 2** 引擎上线: Timo 源码解包 + `CNC_BRAIN_SRC/PY` env 接线 + import 冒烟
- **阶段 3** livekernel 部署: 解包/git clone + pip (镜像源) + settings 外科补丁 + 起服 + 冒烟

## 铁律 (全阶段不变)

1. **data-stays-local**: 笔记本不配置任何云端外发; StepFun 仅 OpenClaw fallback 位
2. **DETERMINISTIC 永远 Timo**: 报价/DFM 数字全部来自确定性引擎, 不经 LLM
3. **显式降级不冒充**: 缺库/缺端点 → 显式 MOCK 标注, 不静默
4. **凭证不出机**: 节点坐标全部 env 注入; 笔记本内无任何密码/IP 字面量

In [ ]:
import os, re, json, time, shlex

# ---- 模式 ----
MODE      = os.environ.get("UEA_BOOT_MODE", "detect")            # detect | execute
TRANSPORT = os.environ.get("UEA_BOOT_TRANSPORT", "local")        # local | ssh
STAGES    = {int(x) for x in os.environ.get("UEA_BOOT_STAGES", "0,1,2,3").split(",") if x.strip()}
SMOKE     = os.environ.get("UEA_BOOT_SMOKE", "0") == "1"         # 推理/引擎/黄金链冒烟
GOLDEN    = os.environ.get("UEA_BOOT_GOLDEN", "0") == "1"        # 黄金链三 verdict (分钟级)

# ---- 节点坐标 (env 注入, 笔记本内无字面量) ----
SSH_HOST = os.environ.get("UEA_NODE_HOST", "")
SSH_PORT = int(os.environ.get("UEA_NODE_PORT", "22"))
SSH_USER = os.environ.get("UEA_NODE_USER", "")
SSH_PWD  = os.environ.get("UEA_NODE_PWD", "")     # 只经 env, 永不打印/落盘
SSH_KEY  = os.environ.get("UEA_NODE_KEY", "")

# ---- 节点布局 (平台标准位置, 可 env 覆盖) ----
OCC_PY         = os.environ.get("UEA_OCC_PY", "~/miniconda3/envs/occ/bin/python")
DEPLOY_DIR     = os.environ.get("UEA_DEPLOY_DIR", "~/union-deploy")
STAGING_DIR    = os.environ.get("UEA_STAGING_DIR", DEPLOY_DIR + "/staging")
BACKUP_DIR     = os.environ.get("UEA_BACKUP_DIR", DEPLOY_DIR + "/backup")
OPENCLAW_CFG   = os.environ.get("UEA_OPENCLAW_CFG", "~/.openclaw/openclaw.json")
GATEWAY_HEALTH = os.environ.get("UEA_GATEWAY_HEALTH", "http://127.0.0.1:18789/health")
VLLM_URL       = os.environ.get("UEA_VLLM_URL", "http://127.0.0.1:8002/v1")
VLLM_MODEL     = os.environ.get("UEA_VLLM_MODEL", "nemotron-omni-30b-a3b")
ENGINE_DIR     = os.environ.get("UEA_ENGINE_DIR", DEPLOY_DIR + "/engine/Timo_CNC-AI-Brain-v12.0-Fusion")
LIVEKERNEL_DIR = os.environ.get("UEA_LIVEKERNEL_DIR", DEPLOY_DIR + "/livekernel")
REPO_URL       = os.environ.get("UEA_REPO_URL", "")               # 开源后可用 git clone 部署
PIP_INDEX      = os.environ.get("UEA_PIP_INDEX", "https://pypi.tuna.tsinghua.edu.cn/simple")
API_PORT       = int(os.environ.get("UEA_API_PORT", "8900"))

WATCH_PORTS = [8002, 8011, 8888, 8900, 8901, 9000, 9100, 18789,
               7862, 8082, 8089, 1234, 1278, 8866]
TS = time.strftime("%Y%m%d-%H%M%S")

print(f"mode={MODE}  transport={TRANSPORT}  stages={sorted(STAGES)}  smoke={SMOKE}  golden={GOLDEN}")
print(f"ssh  = {SSH_USER}@{SSH_HOST}:{SSH_PORT}  (pwd: {'set' if SSH_PWD else 'unset'}, key: {'set' if SSH_KEY else 'unset'})")
print(f"occ  = {OCC_PY}")
print(f"dirs = deploy:{DEPLOY_DIR}  staging:{STAGING_DIR}  engine:{ENGINE_DIR}")
print(f"      livekernel:{LIVEKERNEL_DIR}  backup:{BACKUP_DIR}")

In [ ]:
import subprocess, shutil

class _Report:
    """缺口累加器: 每行 = (域, 项, 状态, 证据, 修复阶段)."""
    def __init__(self):
        self.rows = []
    def add(self, domain, item, status, detail="", fix=""):
        self.rows.append({"domain": domain, "item": item, "status": status,
                          "detail": str(detail)[:400], "fix": fix})
    def counts(self):
        c = {}
        for r in self.rows:
            c[r["status"]] = c.get(r["status"], 0) + 1
        return c
    def gaps(self):
        return [r for r in self.rows if r["status"] in ("fail", "warn")]
    def to_dict(self):
        return {"ts": TS, "mode": MODE, "transport": TRANSPORT,
                "host": sh_out("hostname") if "sh_out" in dir() else "?", "rows": self.rows}

REPORT = _Report()


class LocalX:
    """载体 local: 本机子进程 (bash -lc). 用于笔记本落在节点上直跑."""
    kind = "local"
    def sh(self, cmd, timeout=300):
        print(f"$ {cmd}")
        try:
            p = subprocess.run(["bash", "-lc", cmd], capture_output=True,
                               text=True, timeout=timeout, errors="replace")
            return p.returncode, p.stdout, p.stderr
        except subprocess.TimeoutExpired:
            return 124, "", f"timeout {timeout}s"
    def put(self, local_path, remote_path):
        shutil.copyfile(local_path, remote_path)
    def get(self, remote_path, local_path):
        shutil.copyfile(remote_path, local_path)


class SSHX:
    """载体 ssh: paramiko 驱动远程节点. 只读 detect 也安全 (无破坏性命令)."""
    kind = "ssh"
    def __init__(self):
        import paramiko
        if not SSH_HOST or not SSH_USER or not (SSH_PWD or SSH_KEY):
            raise SystemExit("ssh 载体需要 env: UEA_NODE_HOST / UEA_NODE_USER / (UEA_NODE_PWD | UEA_NODE_KEY)")
        self._c = paramiko.SSHClient()
        self._c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        self._c.connect(SSH_HOST, port=SSH_PORT, username=SSH_USER, password=SSH_PWD or None,
                        key_filename=SSH_KEY or None, timeout=25, banner_timeout=40,
                        auth_timeout=40, allow_agent=False, look_for_keys=False)
        self._sftp = None
    def sh(self, cmd, timeout=300):
        print(f"$ {cmd}")
        try:
            _i, o, e = self._c.exec_command(cmd, timeout=timeout)
            out = o.read().decode("utf-8", "replace")
            err = e.read().decode("utf-8", "replace")
            return o.channel.recv_exit_status(), out, err
        except Exception as ex:
            return 124, "", repr(ex)
    def _sftp_conn(self):
        if self._sftp is None:
            self._sftp = self._c.open_sftp()
        return self._sftp
    def put(self, local_path, remote_path):
        self._sftp_conn().put(local_path, remote_path)
    def get(self, remote_path, local_path):
        self._sftp_conn().get(remote_path, local_path)


X = LocalX() if TRANSPORT == "local" else SSHX()
HOME = X.sh("echo $HOME")[1].strip()
if not HOME.startswith("/"):
    raise SystemExit(f"无法解析远端 $HOME: {HOME!r}")

def _abs(p: str) -> str:
    """把 ~ 开头的路径解析为绝对路径 (远端 $HOME)."""
    return HOME + p[1:] if p.startswith("~") else p

OCC_PY, DEPLOY_DIR, STAGING_DIR, BACKUP_DIR = map(_abs, (OCC_PY, DEPLOY_DIR, STAGING_DIR, BACKUP_DIR))
OPENCLAW_CFG, ENGINE_DIR, LIVEKERNEL_DIR = map(_abs, (OPENCLAW_CFG, ENGINE_DIR, LIVEKERNEL_DIR))
ENGINE_PARENT = ENGINE_DIR.rsplit("/", 1)[0]

def sh(cmd, timeout=300):
    return X.sh(cmd, timeout=timeout)

def sh_out(cmd, timeout=300) -> str:
    rc, out, _ = X.sh(cmd, timeout=timeout)
    return out.strip()

def sh_json(cmd, timeout=300):
    """跑命令并解析 stdout 为 JSON (失败返 None)."""
    rc, out, err = X.sh(cmd, timeout=timeout)
    try:
        return json.loads(out)
    except Exception:
        return None

def curl_json(url, timeout=8):
    return sh_json(f"curl -s -m {timeout} {shlex.quote(url)}")

print(f"transport={X.kind}  HOME={HOME}")

In [ ]:
# ---- DETECT 1/4: 主机 / conda / 端口 (只读) ----
rc, out, _ = sh("uname -srm")
REPORT.add("host", "内核/架构", "info", out.strip())

rc, out, _ = sh("nproc")
REPORT.add("host", "CPU 核数", "info", out.strip())

rc, out, _ = sh(f"df -h {shlex.quote(HOME)} | tail -1")
m = re.search(r"(\d+%)\s+(\S+)$", out.strip())
REPORT.add("host", "磁盘余量", "info" if m else "warn", out.strip(),
           fix="阶段3 部署前清理" if not m else "")

rc, out, _ = sh("free -g 2>/dev/null | awk '/Mem:/{print $7}' || cat /proc/meminfo | awk '/MemAvailable/{print $2/1048576}'")
REPORT.add("host", "可用内存", "info", out.strip() + " GiB")

# conda envs
rc, out, _ = sh(f"ls {HOME}/miniconda3/envs 2>/dev/null || echo NO-MINICONDA")
envs = [e for e in out.split() if e != "NO-MINICONDA"]
REPORT.add("conda", "miniconda envs", "info", ", ".join(envs) or "无")

rc, out, _ = sh(f"{shlex.quote(OCC_PY)} --version 2>&1")
has_occ = rc == 0 and "Python" in out
REPORT.add("conda", "occ env python", "ok" if has_occ else "fail", out.strip(),
           fix="阶段3 前创建 occ 环境 (setup_occ_env.py)" if not has_occ else "")

# occ 依赖盘点 (requirements.txt 口径 + 可选库)
prog = (
    "import importlib.metadata as md, json\n"
    "want=['pyyaml','fastapi','uvicorn','python-multipart','pypdf','openpyxl','reportlab','pytest']\n"
    "got={}\n"
    "for w in want:\n"
    "    try: got[w]=md.version(w)\n"
    "    except Exception: got[w]=None\n"
    "print(json.dumps(got))\n"
)
rc, out, _ = sh(f"{shlex.quote(OCC_PY)} -c {shlex.quote(prog)} 2>&1")
deps = {}
try:
    deps = json.loads(out.strip().splitlines()[-1])
except Exception:
    pass
missing = [k for k, v in deps.items() if v is None]
REPORT.add("conda", "occ 依赖盘点", "ok" if deps and not missing else ("warn" if deps else "fail"),
           json.dumps(deps, ensure_ascii=False) or out.strip()[:200],
           fix="阶段3 pip install -r requirements.txt" if missing else "")

# 端口监听
rc, out, _ = sh("ss -tlnp 2>/dev/null || netstat -tlnp 2>/dev/null")
listening = sorted({int(p) for p in re.findall(r":(\d+)\s", out)})
watch = {p: (p in listening) for p in WATCH_PORTS}
REPORT.add("ports", "关注端口监听态", "info",
           ", ".join(f"{p}:{'UP' if v else '-'}" for p, v in watch.items()))
print(json.dumps(watch))

In [ ]:
# ---- DETECT 2/4: OpenClaw 网关 + vLLM (只读) ----
def _py():
    for cand in ("python3", OCC_PY):
        rc, out, _ = sh(f"{shlex.quote(cand)} --version 2>&1")
        if rc == 0 and "Python" in out:
            return cand
    return OCC_PY

PY_BIN = _py()

rc, out, _ = sh("pgrep -af 'openclaw' | head -5 || true")
gw_procs = [l for l in out.splitlines() if l.strip()]
REPORT.add("openclaw", "网关进程", "ok" if gw_procs else "fail",
           "; ".join(gw_procs)[:200] or "无进程",
           fix="阶段1 前确认 openclaw-gateway 服务存在" if not gw_procs else "")

h = curl_json(GATEWAY_HEALTH, 6)
gw_live = bool((h or {}).get("ok"))
REPORT.add("openclaw", "网关 health", "ok" if gw_live else "fail",
           json.dumps(h, ensure_ascii=False)[:200] if h else "unreachable",
           fix="阶段1 重启网关" if not gw_live else "")

cfg_exists = sh_out(f"test -f {shlex.quote(OPENCLAW_CFG)} && echo YES || echo NO") == "YES"
cfg_summary = None
if cfg_exists:
    prog = (
        "import json,sys;"
        "d=json.load(open(sys.argv[1]));"
        "m=d.get('models') or {};"
        "e=((d.get('agents') or {}).get('entries') or {}).get('main') or {};"
        "mdl=e.get('model') or {};"
        "print(json.dumps({'providers':sorted((m.get('providers') or {}).keys()),"
        "'mode':m.get('mode'),'primary':mdl.get('primary'),"
        "'fallbacks':mdl.get('fallbacks')},ensure_ascii=False))"
    )
    cfg_summary = sh_json(f"{PY_BIN} -c {shlex.quote(prog)} {shlex.quote(OPENCLAW_CFG)}")
primary = (cfg_summary or {}).get("primary") or ""
primary_local = primary.startswith("local-vllm/")
REPORT.add("openclaw", "primary 指向本地 vLLM", "ok" if primary_local else "warn",
           json.dumps(cfg_summary, ensure_ascii=False)[:300] if cfg_summary
           else ("missing" if not cfg_exists else "json 解析失败"),
           fix="阶段1 增 local-vllm provider + 设 primary" if not primary_local else "")

# probe all candidate vLLM endpoints (node may serve on :8000/:8001/:8002)
_ports = []
for _p in ["8000", "8001", "8002", re.search(r":(\d+)", VLLM_URL or "").group(1) if re.search(r":(\d+)", VLLM_URL or "") else ""]:
    if _p and _p not in _ports:
        _ports.append(_p)
_vllm_live = {}
for _p in _ports:
    _m = curl_json(f"http://127.0.0.1:{_p}/v1/models", 6)
    _ids = [x.get("id") for x in (_m or {}).get("data", [])]
    if _ids:
        _vllm_live[_p] = _ids
        REPORT.add("vllm", "endpoint :" + _p + " live", "ok", f"ids={_ids}")
all_ids = [i for _ids in _vllm_live.values() for i in _ids]
serving = VLLM_MODEL in all_ids
REPORT.add("vllm", "本地 vLLM 服务中", "ok" if serving else "fail", f"ids={all_ids}",
           fix="阶段1 前置: 确认 vLLM 起服" if not serving else "")

if SMOKE and serving:
    _ep = next((p for p, _ids in _vllm_live.items() if VLLM_MODEL in _ids), None)
    _url = f"http://127.0.0.1:{_ep}/v1" if _ep else VLLM_URL
    body = json.dumps({"model": VLLM_MODEL,
                       "messages": [{"role": "user", "content": "reply with the single word: pong"}],
                       "max_tokens": 512})
    chat = sh_json(f"curl -s -m 90 {shlex.quote(_url)}/chat/completions "
                   f"-H 'Content-Type: application/json' -d {shlex.quote(body)}")
    ch = ((chat or {}).get("choices") or [{}])[0].get("message") or {}
    content = (ch.get("content") or "").strip()
    reasoning = (ch.get("reasoning") or "").strip()
    ok_smoke = bool(content or reasoning)
    REPORT.add("vllm", "chat 冒烟 (reasoning 格式兼容)", "ok" if ok_smoke else "fail",
               f"content={content[:60]!r} reasoning={reasoning[:60]!r} "
               f"finish={((chat or {}).get('choices') or [{}])[0].get('finish_reason')}",
               fix="放大 max_tokens / 检查 vLLM 启动参数" if not ok_smoke else "")

In [ ]:
# ---- DETECT 3/4: 引擎 / livekernel / staging / 安全 (只读) ----
rc, out, _ = sh(f"test -d {shlex.quote(ENGINE_DIR)} && find {shlex.quote(ENGINE_DIR)} -name '*.py' | wc -l || echo 0")
try:
    py_count = int(out.strip().splitlines()[-1])
except Exception:
    py_count = 0
REPORT.add("engine", "引擎源码目录", "ok" if py_count > 0 else "fail",
           f"{ENGINE_DIR} py={py_count}",
           fix="阶段2 从 staging 解包 / 人工上传引擎包" if py_count == 0 else "")

env_src = sh_out("echo ${CNC_BRAIN_SRC:-unset}")
env_py = sh_out("echo ${CNC_BRAIN_PY:-unset}")
wired = env_src != "unset" and env_py != "unset"
REPORT.add("engine", "CNC_BRAIN_SRC/PY env 接线", "ok" if wired else "warn",
           f"SRC={env_src} PY={env_py}",
           fix="阶段2 写 env.sh + ~/.bashrc 标记块" if not wired else "")

eng_health = curl_json("http://127.0.0.1:7862/api/health", 4)
REPORT.add("engine", "引擎在线服务 :7862", "info" if not eng_health else "ok",
           json.dumps(eng_health, ensure_ascii=False)[:150] if eng_health
           else "无监听 → 走离线桥接 (timo_adapter 自动, 非故障)")

if py_count > 0 and SMOKE:
    rc, out, _ = sh(f"cd {shlex.quote(ENGINE_PARENT)} && PYTHONPATH={shlex.quote(ENGINE_DIR)} "
                    f"{shlex.quote(OCC_PY)} -c \"from src.neuro_core.conflict_check import ConflictChecker; "
                    f"from app.main_lite import calc_quote; print('engine-import-ok')\" 2>&1 | tail -3")
    REPORT.add("engine", "occ python import 引擎", "ok" if "engine-import-ok" in out else "fail",
               out.strip()[:200], fix="核对引擎依赖 (cadquery/OCP 已在 occ)" if "engine-import-ok" not in out else "")

# ---- livekernel ----
lk_exists = sh_out(f"test -d {shlex.quote(LIVEKERNEL_DIR)} && echo YES || echo NO") == "YES"
REPORT.add("livekernel", "部署目录", "ok" if lk_exists else "fail", LIVEKERNEL_DIR,
           fix="阶段3 解包 / git clone" if not lk_exists else "")

if lk_exists:
    rc, out, _ = sh(f"cd {shlex.quote(LIVEKERNEL_DIR)} && {shlex.quote(PY_BIN)} -c \""
                    f"import yaml,json;d=yaml.safe_load(open('config/settings.yaml'));"
                    f"r=d.get('model_router') or {{}};"
                    f"print(json.dumps({{'backend':r.get('backend'),"
                    f"'FAST':(r.get('roles') or {{}}).get('FAST'),"
                    f"engine_src':(d.get('timo') or {{}}).get('engine_src')}},ensure_ascii=False))\" 2>&1")
    try:
        sk = json.loads(out.strip().splitlines()[-1])
    except Exception:
        sk = None
    fast_ep = ((sk or {}).get("FAST") or {}).get("endpoint", "")
    ok_router = ":8002" in fast_ep
    REPORT.add("livekernel", "settings.yaml model_router", "ok" if ok_router else "warn",
               json.dumps(sk, ensure_ascii=False)[:300] if sk else out.strip()[:200],
               fix="阶段3 外科补丁 (→ :8002 Omni-30B)" if not ok_router else "")

# ---- staging ----
rc, out, _ = sh(f"ls -1 {shlex.quote(STAGING_DIR)} 2>/dev/null || echo NO-STAGING")
staging_files = [l for l in out.splitlines() if l.strip() and l.strip() != "NO-STAGING"]
REPORT.add("staging", "暂存区内容", "info", ", ".join(staging_files)[:200] or "空")

# ---- 安全 (P0-A 目标) ----
rc, out, _ = sh(f"find {shlex.quote(DEPLOY_DIR)} -maxdepth 3 \\( -name 'credentials.json' -o -name '.env' \\) 2>/dev/null")
leaks = [l for l in out.splitlines() if l.strip()]
REPORT.add("security", "DEPLOY_DIR 凭据文件", "fail" if leaks else "ok",
           "; ".join(leaks)[:300] or "无", fix="阶段0 整目录替换旧副本" if leaks else "")

old_dir = DEPLOY_DIR + "/union-export-demo"
old_tar = DEPLOY_DIR + "/union-export-node.tar.gz"
has_old = sh_out(f"test -e {shlex.quote(old_dir)} && echo YES || echo NO") == "YES"
has_tar = sh_out(f"test -f {shlex.quote(old_tar)} && echo YES || echo NO") == "YES"
REPORT.add("security", "旧副本/旧 tarball", "fail" if (has_old or has_tar) else "ok",
           f"dir={has_old} tar={has_tar}",
           fix="阶段0 备份后 rm -rf" if (has_old or has_tar) else "")

## REASON — 缺口汇总

上一格起四个 DETECT 区块已把全域状态写进 `REPORT`。本格汇总 fail/warn 缺口并给出修复计划;
`execute` 模式下随后每个阶段会**先复检再动作**, 已完成的不重复做 (幂等)。

In [ ]:
# ---- REASON: 缺口汇总 + 修复计划 (只读) ----
STAGE_NAMES = {0: "阶段0 安全止血 (旧副本替换)", 1: "阶段1 OpenClaw 接线 (本地 Omni-30B primary)",
               2: "阶段2 引擎上线 (Timo 源码 + env)", 3: "阶段3 livekernel 部署 (解包+起服)"}

print("=" * 76)
print(f"检测汇总  mode={MODE} transport={X.kind}  {json.dumps(REPORT.counts(), ensure_ascii=False)}")
print("=" * 76)
MARK = {"ok": "[ok]  ", "warn": "[warn]", "fail": "[FAIL]", "info": "[info]", "skip": "[skip]"}
for r in REPORT.rows:
    print(f"{MARK.get(r['status'], '[?]   ')} {r['domain']:<11}{r['item']}  {r['detail'][:150]}")

gaps = REPORT.gaps()
print("-" * 76)
if gaps:
    print(f"缺口 {len(gaps)} 项 (fail={sum(1 for g in gaps if g['status']=='fail')}, "
          f"warn={sum(1 for g in gaps if g['status']=='warn')}):")
    for g in gaps:
        print(f"  - [{g['status']}] {g['domain']}/{g['item']}  ->  {g['fix'] or '人工评估'}")
else:
    print("无 fail/warn 缺口。")

print("-" * 76)
if MODE == "detect":
    print("当前为 detect (只读) 模式。修复计划 (execute 模式将按序执行):")
    for s in sorted(STAGES):
        print(f"  {STAGE_NAMES.get(s, s)}")
    print("切换: UEA_BOOT_MODE=execute [UEA_BOOT_STAGES=0,1,2,3] [UEA_BOOT_SMOKE=1]")
else:
    print(f"execute 模式, 待执行阶段: {[STAGE_NAMES.get(s, s) for s in sorted(STAGES)]}")

try:
    from IPython.display import Markdown, display
    lines = ["| 域 | 项 | 状态 | 证据 |", "|---|---|---|---|"]
    for r in REPORT.rows:
        lines.append(f"| {r['domain']} | {r['item']} | {r['status']} | {r['detail'][:80]} |")
    display(Markdown("\n".join(lines)))
except Exception:
    pass

## EXECUTE · 破坏性阶段的守卫契约

`execute` 模式按 `UEA_BOOT_STAGES` 逐阶段执行; 每个阶段:

1. **先留证后动手** — `rm -rf` 只允许 `ALLOWED_RM` 精确条目, 删前 `ls -ld` 打印;
2. **可逆优先** — 凭据/旧文件一律 `quarantine()` 移入 `BACKUP_DIR/quarantine-<TS>` (chmod 600), 不直接删;
3. **显式降级** — 缺包/缺文件记 `REPORT` fail 行 + 修复提示, **不静默不冒充**;
4. **阶段末复扫** — 每个阶段结尾重新探测, 把结果写回 `REPORT` (验收证据)。

重启网关 / pip 安装 / 整目录替换三个"重动作"由独立 env 开关控制, 默认全关。

In [ ]:
# ---- EXECUTE 守卫: 模式开关 + 破坏性操作白名单 (PRD 风险表: 误删节点目录) ----
EXEC           = (MODE == "execute")
RESTART_GW     = os.environ.get("UEA_BOOT_RESTART_GW", "0") == "1"     # 阶段1 重启网关
PIP_INSTALL    = os.environ.get("UEA_BOOT_PIP_INSTALL", "0") == "1"    # 阶段2/3 pip 补依赖
FORCE_REPLACE  = os.environ.get("UEA_BOOT_FORCE_REPLACE", "0") == "1"  # 阶段3 整目录替换
QUARANTINE_DIR = BACKUP_DIR + "/quarantine-" + TS

# rm -rf 白名单: 仅两条旧副本 + 阶段3 的 livekernel 部署目录
ALLOWED_RM = {
    _abs(DEPLOY_DIR + "/union-export-demo"),
    _abs(DEPLOY_DIR + "/union-export-node.tar.gz"),
    _abs(LIVEKERNEL_DIR),
}

def stage_enabled(n):
    return EXEC and n in STAGES

def _assert_allowed(path, allowed=None):
    p = _abs(path)
    allowed = allowed or ALLOWED_RM
    if p not in allowed:
        raise SystemExit("拒绝 rm: %s 不在白名单 %s" % (p, sorted(allowed)))
    if not p.startswith(HOME + "/"):
        raise SystemExit("拒绝 rm: %s 不在 HOME 下" % p)
    return p

def rm_rf_exact(path):
    # 删前 ls 精确留证 (PRD 风险表: rm 前 ls 精确匹配, 只动 ~/union-deploy 两条目)
    p = _assert_allowed(path)
    rc, out, _ = sh("ls -ld %s 2>/dev/null || echo MISSING" % shlex.quote(p))
    print("  rm 目标留证: " + out.strip())
    if "MISSING" in out:
        return False
    rc, out, err = sh("rm -rf %s" % shlex.quote(p))
    if rc != 0:
        raise SystemExit("rm -rf 失败 %s: %s" % (p, err[:200]))
    print("  已删: " + p)
    return True

def quarantine(path, why):
    # 凭证/旧文件移入隔离区 (可逆), 不直接删
    p = _abs(path)
    sh("mkdir -p %s" % shlex.quote(QUARANTINE_DIR))
    dst = QUARANTINE_DIR + "/" + p.strip("/").replace("/", "__")
    rc, out, err = sh("mv %s %s && chmod 600 %s"
                      % (shlex.quote(p), shlex.quote(dst), shlex.quote(dst)))
    if rc != 0:
        REPORT.add("security", "隔离失败 " + p, "fail", err[:200],
                   fix="人工 mv 至 " + QUARANTINE_DIR)
        return ""
    REPORT.add("security", "已隔离 (" + why + ")", "ok", p + " -> " + dst)
    return dst

def _write_local(name, content):
    import tempfile
    d = tempfile.mkdtemp(prefix="uea-boot-")
    p = d + "/" + name
    with open(p, "w", encoding="utf-8", newline="\n") as f:
        f.write(content)
    return p

def run_script(name, content, args="", runner=None, remote_dir=None):
    # 本地生成脚本 -> put 到 staging -> 节点执行 (避免 heredoc 引号地狱)
    lp = _write_local(name, content)
    rdir = _abs(remote_dir or STAGING_DIR)
    rp = rdir + "/" + name
    sh("mkdir -p %s" % shlex.quote(rdir))
    X.put(lp, rp)
    return sh("cd %s && %s %s %s"
              % (shlex.quote(rdir), runner or PY_BIN, shlex.quote(rp), args))

print("execute=%s  stages=%s  restart_gw=%s  pip=%s  force_replace=%s"
      % (EXEC, sorted(STAGES), RESTART_GW, PIP_INSTALL, FORCE_REPLACE))
print("rm 白名单:")
for p in sorted(ALLOWED_RM):
    print("  - " + p)

In [ ]:
# ---- EXECUTE 阶段0: 安全止血 (PRD 第4节; 只动 ~/union-deploy 两条目) ----
if not stage_enabled(0):
    print("跳过阶段0 (detect 模式或 UEA_BOOT_STAGES 未含 0)")
else:
    # 1) 凭据隔离 (可逆): credentials.json / .env
    rc, out, _ = sh("find %s -maxdepth 3 \\( -name 'credentials.json' -o -name '.env' \\) 2>/dev/null"
                    % shlex.quote(DEPLOY_DIR))
    leaks = [l for l in out.splitlines() if l.strip()]
    for f in leaks:
        quarantine(f, "DEPLOY_DIR 凭据")
    # 2) 本地备份提醒 (PRD 4.1: 备份只留本地 Windows, 含 credentials.json 供轮换核对, 内容不示)
    REPORT.add("security", "本地备份旧副本", "info",
               "PRD 4.1: 部署方在本地 tar 旧副本 (文件名可示/内容不示), 笔记本不代传")
    # 3) rm -rf 两条目 (删前 ls 留证; 不碰 ~/inference ~/nvidia ~/miniconda3)
    for p in (DEPLOY_DIR + "/union-export-demo", DEPLOY_DIR + "/union-export-node.tar.gz"):
        rm_rf_exact(p)
    # 4) 复扫验证
    rc, out, _ = sh("find %s -maxdepth 3 \\( -name 'credentials.json' -o -name '.env' \\) 2>/dev/null"
                    % shlex.quote(DEPLOY_DIR))
    left = [l for l in out.splitlines() if l.strip()]
    REPORT.add("security", "DEPLOY_DIR 凭据复扫", "ok" if not left else "fail",
               "; ".join(left)[:300] or "无残留")
    rc, out, _ = sh("ls -1 %s 2>/dev/null" % shlex.quote(DEPLOY_DIR))
    REPORT.add("security", "DEPLOY_DIR 顶层", "info",
               ", ".join(l for l in out.splitlines() if l.strip())[:300] or "空")

In [ ]:
# ---- EXECUTE 阶段1: OpenClaw 接线 (PRD 第5节: 本地 Omni-30B 设 primary, StepFun 留 fallback) ----
if not stage_enabled(1):
    print("跳过阶段1 (detect 模式或 UEA_BOOT_STAGES 未含 1)")
else:
    if sh_out("test -f %s && echo YES || echo NO" % shlex.quote(OPENCLAW_CFG)) != "YES":
        REPORT.add("openclaw", "配置文件", "fail", OPENCLAW_CFG + " 不存在",
                   fix="确认 openclaw 安装路径 (UEA_OPENCLAW_CFG 可覆盖)")
    else:
        # 1) 备份: 节点本地 .bak (PRD 回滚路径) + BACKUP_DIR 双份
        bak_local = OPENCLAW_CFG + ".bak-" + TS
        bak_dir = BACKUP_DIR + "/openclaw.json.bak-" + TS
        sh("mkdir -p %s && cp %s %s && cp %s %s"
           % (shlex.quote(BACKUP_DIR), shlex.quote(OPENCLAW_CFG), shlex.quote(bak_local),
              shlex.quote(OPENCLAW_CFG), shlex.quote(bak_dir)))
        REPORT.add("openclaw", "配置备份", "ok", bak_local + " ; " + bak_dir)

        # 2) 外科补丁 (PRD 5.2: baseUrl/api/models; primary; fallbacks 原序不动; mode 不动)
        PATCH_SRC = """# patch_openclaw.py -- PRD 5.2: local-vllm provider + primary (fallbacks/mode 不动)
import json, os, shutil, sys, time

cfg, vllm_url, vllm_model, action = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4]
with open(cfg, encoding="utf-8") as f:
    d = json.load(f)  # 解析失败即语法坏, 不写回 (PRD 风险表)

models = d.setdefault("models", {})
provs = models.setdefault("providers", {})
provs["local-vllm"] = {"baseUrl": vllm_url, "api": "openai-completions",
                       "models": [vllm_model]}
mdl = d.setdefault("agents", {}).setdefault("entries", {}).setdefault("main", {}).setdefault("model", {})
old_primary = mdl.get("primary")
mdl["primary"] = "local-vllm/" + vllm_model
# fallbacks 原序保留 (D2: 无 key 自然不生效); models.mode 保持 merge 不动

out = {"old_primary": old_primary, "new_primary": mdl["primary"],
       "fallbacks": mdl.get("fallbacks"), "mode": models.get("mode"),
       "providers": sorted(provs.keys())}
if action != "apply":
    out["dry"] = True
    print(json.dumps(out, ensure_ascii=False))
    sys.exit(0)

bak = cfg + ".bak-uea-" + time.strftime("%Y%m%d-%H%M%S")
shutil.copy2(cfg, bak)
tmp = cfg + ".tmp-uea"
with open(tmp, "w", encoding="utf-8") as f:
    json.dump(d, f, ensure_ascii=False, indent=2)
with open(tmp, encoding="utf-8") as f:
    json.load(f)  # 写后重读校验
os.replace(tmp, cfg)
out["bak"] = bak
print(json.dumps(out, ensure_ascii=False))
"""
        rc, out, err = run_script("patch_openclaw.py", PATCH_SRC,
                                  args=" ".join(shlex.quote(x) for x in
                                                (OPENCLAW_CFG, VLLM_URL, VLLM_MODEL, "apply")))
        try:
            pr = json.loads(out.strip().splitlines()[-1])
        except Exception:
            pr = None
        ok = bool(pr) and pr.get("new_primary") == "local-vllm/" + VLLM_MODEL
        REPORT.add("openclaw", "openclaw.json 补丁", "ok" if ok else "fail",
                   json.dumps(pr, ensure_ascii=False)[:300] if pr else (out + err)[:300],
                   fix="还原 .bak 后人工编辑" if not ok else "")
        rc2, out2, _ = sh("%s -m json.tool %s > /dev/null && echo JSON-OK || echo JSON-BAD"
                          % (PY_BIN, shlex.quote(OPENCLAW_CFG)))
        REPORT.add("openclaw", "json.tool 语法复验", "ok" if "JSON-OK" in out2 else "fail",
                   out2.strip()[:120])

        # 3) 停垫片 :8901 (PRD 5.3); 不动上游 :8888 inference api_server
        rc, out, _ = sh("ss -tlnp 2>/dev/null | grep ':8901 ' || true")
        pids = sorted(set(re.findall(r"pid=(\d+)", out)))
        if pids:
            sh("kill " + " ".join(pids))
            time.sleep(1.0)
        rc, out, _ = sh("ss -tln 2>/dev/null | grep -c ':8901 ' || true")
        still = out.strip() not in ("", "0")
        REPORT.add("openclaw", "垫片 :8901 停止", "ok" if not still else "warn",
                   "killed pids=" + (",".join(pids) or "无"),
                   fix="人工 kill 残留 pid" if still else "")
        rc, out, _ = sh("ss -tln 2>/dev/null | grep -c ':8888 ' || true")
        REPORT.add("openclaw", "inference :8888 不动", "info", "listening=" + out.strip())

        # 4) 重启网关 (重动作, 默认关; UEA_BOOT_RESTART_GW=1 才做)
        if RESTART_GW:
            sh("systemctl --user restart openclaw-gateway.service 2>&1 || "
               "systemctl restart openclaw-gateway 2>&1")
            live, h = False, None
            for _ in range(30):
                h = curl_json(GATEWAY_HEALTH, 4)
                if (h or {}).get("ok"):
                    live = True
                    break
                time.sleep(1)
            REPORT.add("openclaw", "网关重启 + health", "ok" if live else "fail",
                       json.dumps(h, ensure_ascii=False)[:150] if h else "unreachable",
                       fix="还原 .bak + 重启" if not live else "")
        else:
            REPORT.add("openclaw", "网关重启", "skip",
                       "UEA_BOOT_RESTART_GW=1 才重启; 配置已写, 重启后生效")

        # 5) 冒烟: 直接打 vLLM, 覆盖 PRD 3.4 reasoning 格式 (content 可能为 null)
        if SMOKE:
            body = json.dumps({"model": VLLM_MODEL,
                               "messages": [{"role": "user", "content": "用一句话介绍你自己"}],
                               "max_tokens": 512, "temperature": 0.2})
            rc, out, err = sh("curl -s -m 90 %s -H 'Content-Type: application/json' -d %s"
                              % (shlex.quote(VLLM_URL + "/chat/completions"), shlex.quote(body)))
            text, mid = "", ""
            try:
                r = json.loads(out)
                msg = (r.get("choices") or [{}])[0].get("message") or {}
                text = (msg.get("reasoning") or msg.get("content") or "").strip()
                mid = r.get("model") or ""
            except Exception:
                text = ""
            REPORT.add("openclaw", "vLLM 冒烟 (reasoning 格式)", "ok" if text else "fail",
                       "model=%s len=%d %s" % (mid, len(text), text[:80]),
                       fix="检查 :8002 与 max_tokens (PRD 3.4)" if not text else "")
        else:
            REPORT.add("openclaw", "vLLM 冒烟", "skip", "UEA_BOOT_SMOKE=1 开启")

In [ ]:
# ---- EXECUTE 阶段2: 引擎上线 (PRD 第6节: Timo 源码落节点 + env 接线 + 冒烟) ----
if not stage_enabled(2):
    print("跳过阶段2 (detect 模式或 UEA_BOOT_STAGES 未含 2)")
else:
    ENGINE_BACKUP = BACKUP_DIR + "/engine-old-" + TS

    def _engine_py_count():
        rc, out, _ = sh("test -d %s && find %s -name '*.py' 2>/dev/null | wc -l || echo 0"
                        % (shlex.quote(ENGINE_DIR), shlex.quote(ENGINE_DIR)))
        try:
            return int(out.strip().splitlines()[-1])
        except Exception:
            return 0

    def _rm_extract(p):
        # 只允许删 ENGINE_PARENT 下 .extract- 前缀临时解包目录 (前缀断言, 同白名单思想)
        p = _abs(p)
        if not (p.startswith(ENGINE_PARENT + "/.extract-") and p.startswith(HOME + "/")):
            raise SystemExit("拒绝 rm: " + p)
        sh("rm -rf %s" % shlex.quote(p))

    py_now = _engine_py_count()
    if py_now >= 44:
        REPORT.add("engine", "引擎已部署", "ok", "%s py=%d" % (ENGINE_DIR, py_now))
    else:
        rc, out, _ = sh("ls -1 %s 2>/dev/null || true" % shlex.quote(STAGING_DIR))
        cands = [l for l in out.splitlines()
                 if re.search(r"(?i)(timo|engine)", l)
                 and re.search(r"\.(tar\.gz|tgz|tar\.zst|zip)$", l)]
        if len(cands) != 1:
            REPORT.add("engine", "staging 引擎包", "fail", "候选=%s" % cands,
                       fix="按 PRD 6.1 打包引擎源码 (app/+src/+requirements*.txt) 传入 staging")
        else:
            tarball = STAGING_DIR + "/" + cands[0]
            ex_dir = ENGINE_PARENT + "/.extract-" + TS
            sh("mkdir -p %s" % shlex.quote(ex_dir))
            rc, out, err = sh("tar -xf %s -C %s 2>&1 || zstd -dc %s | tar -x -C %s 2>&1"
                              % (shlex.quote(tarball), shlex.quote(ex_dir),
                                 shlex.quote(tarball), shlex.quote(ex_dir)))
            rc2, out2, _ = sh("find %s -maxdepth 3 -name 'main_lite.py' -path '*/app/*' | head -1"
                              % shlex.quote(ex_dir))
            found = out2.strip()
            if not found:
                REPORT.add("engine", "解包定位", "fail", "未找到 app/main_lite.py: " + err[:150],
                           fix="核对引擎包结构 (需 app/main_lite.py)")
            else:
                top = found[len(ex_dir):].lstrip("/").split("/")[0]
                src_dir = ex_dir + "/" + top
                if top != ENGINE_DIR.rsplit("/", 1)[1]:
                    REPORT.add("engine", "目录名与 UEA_ENGINE_DIR 不一致", "warn",
                               "解包得 %s, 期望 %s" % (top, ENGINE_DIR.rsplit("/", 1)[1]),
                               fix="将 mv 到 UEA_ENGINE_DIR 指定名")
                if sh_out("test -d %s && echo YES || echo NO" % shlex.quote(ENGINE_DIR)) == "YES":
                    if not FORCE_REPLACE:
                        REPORT.add("engine", "目标目录已存在", "fail", ENGINE_DIR,
                                   fix="UEA_BOOT_FORCE_REPLACE=1 整目录替换 (自动先备份)")
                        src_dir = None
                    else:
                        sh("mkdir -p %s && mv %s %s"
                           % (shlex.quote(ENGINE_BACKUP), shlex.quote(ENGINE_DIR),
                              shlex.quote(ENGINE_BACKUP)))
                        REPORT.add("engine", "旧引擎已备份", "ok", ENGINE_BACKUP)
                if src_dir:
                    rc, out, err = sh("mv %s %s" % (shlex.quote(src_dir), shlex.quote(ENGINE_DIR)))
                    if rc != 0:
                        REPORT.add("engine", "落位", "fail", err[:200], fix="人工 mv 核对")
                    else:
                        _rm_extract(ex_dir)
                        py_after = _engine_py_count()
                        REPORT.add("engine", "引擎落位", "ok" if py_after > 0 else "fail",
                                   "%s py=%d (PRD 验收 44)" % (ENGINE_DIR, py_after))

    # ---- env 接线 (PRD 6.3): env.sh + ~/.bashrc 标记块, 幂等 ----
    ENV_SH = DEPLOY_DIR + "/env.sh"
    env_src = ('# UEA engine env -- generated by node_bootstrap.ipynb (阶段2)\n'
               'export CNC_BRAIN_SRC="%s"\n'
               'export CNC_BRAIN_PY="%s"\n' % (ENGINE_DIR, OCC_PY))
    lp = _write_local("env.sh", env_src)
    X.put(lp, ENV_SH)
    sh("chmod 600 %s" % shlex.quote(ENV_SH))
    WIRE_SRC = r"""# wire_env.sh -- env.sh + ~/.bashrc 标记块 (幂等)
set -euo pipefail
ENV_SH="$1"; BASHRC="$HOME/.bashrc"
BEGIN="# BEGIN UEA-ENGINE-ENV"; END="# END UEA-ENGINE-ENV"
touch "$BASHRC"
if grep -qF "$BEGIN" "$BASHRC"; then
  sed -i "\|$BEGIN|,\|$END|d" "$BASHRC"
fi
printf '
%s
[ -f "%s" ] && . "%s"
%s
' "$BEGIN" "$ENV_SH" "$ENV_SH" "$END" >> "$BASHRC"
echo "--- bashrc block ---"
grep -A1 -B1 "$BEGIN" "$BASHRC"
"""
    rc, out, err = run_script("wire_env.sh", WIRE_SRC, args=shlex.quote(ENV_SH), runner="bash")
    REPORT.add("engine", "env.sh + bashrc 标记块", "ok" if rc == 0 else "fail",
               out.strip()[-300:], fix="人工追加标记块" if rc != 0 else "")
    # 当前会话内联生效 (供本笔记本后续冒烟使用)
    os.environ["CNC_BRAIN_SRC"] = ENGINE_DIR
    os.environ["CNC_BRAIN_PY"] = OCC_PY

    # ---- 冒烟: occ python import 引擎 (PRD 6.5) ----
    if SMOKE and sh_out("test -d %s && echo YES || echo NO" % shlex.quote(ENGINE_DIR)) == "YES":
        rc, out, _ = sh("PYTHONPATH=%s %s -c \"from src.neuro_core.conflict_check import "
                        "ConflictChecker; from app.main_lite import calc_quote; "
                        "print('engine-import-ok')\" 2>&1 | tail -3"
                        % (shlex.quote(ENGINE_DIR), shlex.quote(OCC_PY)))
        REPORT.add("engine", "occ python import 引擎", "ok" if "engine-import-ok" in out else "fail",
                   out.strip()[:300],
                   fix="核对引擎依赖 (cadquery/OCP 已在 occ); aarch64 差异记录不冒充" if "engine-import-ok" not in out else "")

    # ---- 依赖核对 (PRD 6.6): 缺项报告, pip 补装默认关 ----
    req = ENGINE_DIR + "/requirements.txt"
    if sh_out("test -f %s && echo YES || echo NO" % shlex.quote(req)) == "YES":
        REQ_SRC = """# check_reqs.py -- 引擎 requirements vs occ env 已装包
import importlib.metadata as md, re, sys
miss = []
for l in open(sys.argv[1], encoding="utf-8"):
    l = l.split("#")[0].strip()
    if not l:
        continue
    m = re.match(r"[A-Za-z0-9_.-]+", l)
    if not m:
        continue
    base = m.group(0).lower().replace("_", "-")
    try:
        md.version(base)
    except Exception:
        miss.append(l)
print("MISSING:" + "|".join(miss) if miss else "ALL-PRESENT")
"""
        rc, out, err = run_script("check_reqs.py", REQ_SRC, args=shlex.quote(req))
        missing = [] if "ALL-PRESENT" in out else [m for m in out.split("MISSING:")[-1].split("|") if m.strip()]
        REPORT.add("engine", "requirements 核对", "ok" if not missing else "warn",
                   ("missing=" + ",".join(missing)) if missing else "ALL-PRESENT",
                   fix="UEA_BOOT_PIP_INSTALL=1 走清华源补装" if missing else "")
        if missing and PIP_INSTALL:
            rc, out, err = sh("%s -m pip install -r %s -i %s 2>&1 | tail -5"
                              % (shlex.quote(OCC_PY), shlex.quote(req), shlex.quote(PIP_INDEX)))
            REPORT.add("engine", "pip 补依赖", "ok" if rc == 0 else "fail", out.strip()[:300])

In [ ]:
# ---- EXECUTE 阶段3: livekernel 上线 (源码落位 + settings 外科补丁 + 起服 + 冒烟) ----
if not stage_enabled(3):
    print("跳过阶段3 (detect 模式或 UEA_BOOT_STAGES 未含 3)")
else:
    def _rm_extract_lk(p):
        # 只允许删 DEPLOY_DIR 下 .extract-lk- 前缀临时解包目录 (前缀断言)
        p = _abs(p)
        if not (p.startswith(DEPLOY_DIR + "/.extract-lk-") and p.startswith(HOME + "/")):
            raise SystemExit("拒绝 rm: " + p)
        sh("rm -rf %s" % shlex.quote(p))

    SETTINGS = LIVEKERNEL_DIR + "/config/settings.yaml"
    lk_present = sh_out("test -d %s && echo YES || echo NO" % shlex.quote(LIVEKERNEL_DIR)) == "YES"
    if lk_present and not FORCE_REPLACE:
        REPORT.add("livekernel", "部署目录已存在", "info",
                   LIVEKERNEL_DIR + " (FORCE_REPLACE=1 才替换)")
    else:
        # 源码优先级: staging tarball > REPO_URL git clone > 缺口
        rc, out, _ = sh("ls -1 %s 2>/dev/null || true" % shlex.quote(STAGING_DIR))
        cands = [l for l in out.splitlines()
                 if re.search(r"(?i)(livekernel|union-export|node-v631)", l)
                 and re.search(r"\.(tar\.gz|tgz)$", l)]
        got = False
        if cands:
            tarball = STAGING_DIR + "/" + cands[0]
            ex_dir = DEPLOY_DIR + "/.extract-lk-" + TS
            sh("mkdir -p %s && tar -xzf %s -C %s"
               % (shlex.quote(ex_dir), shlex.quote(tarball), shlex.quote(ex_dir)))
            rc2, out2, _ = sh("find %s -maxdepth 3 -name 'api_server.py' -path '*/services/*' | head -1"
                              % shlex.quote(ex_dir))
            found = out2.strip()
            if found:
                top = found[len(ex_dir):].lstrip("/").split("/")[0]
                if lk_present:
                    q = BACKUP_DIR + "/livekernel-old-" + TS
                    sh("mkdir -p %s && mv %s %s"
                       % (shlex.quote(BACKUP_DIR), shlex.quote(LIVEKERNEL_DIR), shlex.quote(q)))
                    REPORT.add("livekernel", "旧部署已备份", "ok", q)
                rc, out, err = sh("mv %s %s" % (shlex.quote(ex_dir + "/" + top),
                                                shlex.quote(LIVEKERNEL_DIR)))
                got = rc == 0
                _rm_extract_lk(ex_dir)
                REPORT.add("livekernel", "staging 解包落位", "ok" if got else "fail",
                           "%s -> %s" % (cands[0], LIVEKERNEL_DIR))
            else:
                _rm_extract_lk(ex_dir)
                REPORT.add("livekernel", "staging 解包定位", "fail", "未找到 services/api_server.py",
                           fix="核对 node tarball 结构")
        if not got and REPO_URL:
            rc, out, err = sh("git clone --depth 1 %s %s 2>&1 | tail -3"
                              % (shlex.quote(REPO_URL), shlex.quote(LIVEKERNEL_DIR)))
            got = rc == 0
            REPORT.add("livekernel", "git clone", "ok" if got else "fail", out.strip()[:200])
        if not got:
            REPORT.add("livekernel", "源码获取", "fail", "staging 无 livekernel 包, REPO_URL 未设",
                       fix="上传 node tarball 至 staging 或设 UEA_REPO_URL")

    # ---- 验证 + settings 外科补丁 + 起服 (目录就绪才继续) ----
    if sh_out("test -f %s && echo YES || echo NO" % shlex.quote(SETTINGS)) == "YES":
        rc, out, _ = sh("find %s -name '*.py' 2>/dev/null | wc -l" % shlex.quote(LIVEKERNEL_DIR))
        REPORT.add("livekernel", "源码落位", "info", "py=" + out.strip())

        rc, out, _ = sh("find %s -maxdepth 2 \\( -name 'credentials.json' -o -name '.env' \\) 2>/dev/null"
                        % shlex.quote(LIVEKERNEL_DIR))
        leaks = [l for l in out.splitlines() if l.strip()]
        REPORT.add("security", "livekernel 树凭据复扫", "ok" if not leaks else "fail",
                   "; ".join(leaks)[:200] or "无 (导出包铁律)")

        # settings 外科补丁 (PRD 3.3; 行级正则保注释; 改前备份, 改后 yaml 校验)
        emb = curl_json("http://127.0.0.1:8011/v1/models", 4)
        emb_ids = [m.get("id") for m in (emb or {}).get("data", []) if isinstance(m, dict)]
        embed_ep = "http://127.0.0.1:8011/v1" if emb_ids else ""
        embed_model = emb_ids[0] if len(emb_ids) == 1 else ""
        PATCH_SET = r"""# patch_settings.py -- PRD 3.3 角色映射 (行级正则, 保注释; 改前备份, 改后 yaml 校验)
import os, re, shutil, sys, time

cfg, fast_ep, fast_model, embed_ep, embed_model, eng_src, eng_py = sys.argv[1:8]
src = open(cfg, encoding="utf-8").read()

def sub_role(text, role, ep, model=None):
    pat = re.compile(r"(^\s*" + role + r":\s*\{\s*endpoint:\s*)([\"'])[^\"']*\2"
                     r"(\s*,\s*model:\s*)([\"'])([^\"']*)\4(\s*\})", re.M)
    def _rep(m):
        mdl = m.group(5) if model is None else model
        return m.group(1) + chr(34) + ep + chr(34) + m.group(3) + m.group(4) + mdl + m.group(4) + m.group(6)
    return pat.sub(_rep, text, count=1)

new = src
for role in ("FAST", "VISION", "REASON"):
    new = sub_role(new, role, fast_ep, fast_model)
if embed_ep:
    new = sub_role(new, "EMBED", embed_ep, embed_model or None)
new = re.sub(r"(^\s*engine_src:\s*)([\"'])[^\"']*\2",
             lambda m: m.group(1) + chr(34) + eng_src + chr(34), new, count=1, flags=re.M)
new = re.sub(r"(^\s*engine_python:\s*)([\"'])[^\"']*\2",
             lambda m: m.group(1) + chr(34) + eng_py + chr(34), new, count=1, flags=re.M)

if new == src:
    print("NO-CHANGE")
    sys.exit(0)

bak = cfg + ".bak-uea-" + time.strftime("%Y%m%d-%H%M%S")
shutil.copy2(cfg, bak)
tmp = cfg + ".tmp-uea"
open(tmp, "w", encoding="utf-8").write(new)
try:
    import yaml
    yaml.safe_load(open(tmp, encoding="utf-8"))
    note = "yaml-ok"
except ImportError:
    note = "yaml-skip(PyYAML 缺失, 未校验)"
os.replace(tmp, cfg)
print("PATCHED " + bak + " " + note)
bl, nl = src.splitlines(), new.splitlines()
for i in range(max(len(bl), len(nl))):
    b = bl[i] if i < len(bl) else "<none>"
    n = nl[i] if i < len(nl) else "<none>"
    if b != n:
        print("  - " + b.strip() + "  =>  " + n.strip())
"""
        rc, out, err = run_script("patch_settings.py", PATCH_SET,
                                  args=" ".join(shlex.quote(x) for x in
                                                (SETTINGS, VLLM_URL, VLLM_MODEL,
                                                 embed_ep, embed_model, ENGINE_DIR, OCC_PY)))
        ok = ("PATCHED" in out) or ("NO-CHANGE" in out)
        REPORT.add("livekernel", "settings.yaml 外科补丁", "ok" if ok else "fail",
                   out.strip()[:400], fix="还原 .bak-uea-* 后人工编辑" if not ok else "")
        rc, out, _ = sh("%s -c \"import yaml,json;d=yaml.safe_load(open(%r));"
                        "r=d['model_router']['roles'];"
                        "print(json.dumps({'FAST':r['FAST'],'EMBED':r['EMBED'],"
                        "'engine_src':d['timo']['engine_src']},ensure_ascii=False))\" 2>&1"
                        % (PY_BIN, SETTINGS))
        REPORT.add("livekernel", "settings 复验", "info", out.strip()[:300])

        # pip 依赖 (重动作, 默认关; occ 已有 fastapi/uvicorn/pypdf/openpyxl/pytest)
        req = LIVEKERNEL_DIR + "/requirements.txt"
        if PIP_INSTALL and sh_out("test -f %s && echo YES || echo NO" % shlex.quote(req)) == "YES":
            rc, out, err = sh("%s -m pip install -r %s -i %s 2>&1 | tail -6"
                              % (shlex.quote(OCC_PY), shlex.quote(req), shlex.quote(PIP_INDEX)))
            REPORT.add("livekernel", "pip 依赖安装", "ok" if rc == 0 else "fail", out.strip()[:300])
        else:
            REPORT.add("livekernel", "pip 依赖安装", "skip",
                       "UEA_BOOT_PIP_INSTALL=1 才装 (reportlab 缺则报价 PDF 显式降级不炸链)")

        # 起服 (后台) + health 轮询
        logf = DEPLOY_DIR + "/api-" + TS + ".log"
        sh("cd %s && nohup %s services/api_server.py --port %d > %s 2>&1 & sleep 1"
           % (shlex.quote(LIVEKERNEL_DIR), shlex.quote(OCC_PY), API_PORT, shlex.quote(logf)))
        live, h = False, None
        for _ in range(40):
            h = curl_json("http://127.0.0.1:%d/health" % API_PORT, 4)
            if (h or {}).get("status") == "ok":
                live = True
                break
            time.sleep(1)
        REPORT.add("livekernel", "API 起服 :%d" % API_PORT, "ok" if live else "fail",
                   json.dumps(h, ensure_ascii=False)[:200] if h else "见日志 " + logf,
                   fix="tail -50 " + logf if not live else "")

        # 黄金链 (进程内 --offline, 不依赖 API; 分钟级)
        if GOLDEN:
            rc, out, err = sh("cd %s && %s scripts/run_golden_core.py --offline 2>&1 | tail -25"
                              % (shlex.quote(LIVEKERNEL_DIR), shlex.quote(OCC_PY)))
            REPORT.add("livekernel", "黄金链 --offline", "info" if rc == 0 else "warn",
                       out.strip()[-500:], fix="核对输出 verdict" if rc != 0 else "")
        else:
            REPORT.add("livekernel", "黄金链 --offline", "skip", "UEA_BOOT_GOLDEN=1 开启")

In [ ]:
# ---- FINAL: 证据落盘 + 验收汇总 (PRD 第7节) ----
doc = REPORT.to_dict()
doc["stages_enabled"] = sorted(STAGES)
doc["gaps"] = REPORT.gaps()

print("=" * 76)
print("最终汇总  mode=%s  transport=%s  %s"
      % (MODE, X.kind, json.dumps(REPORT.counts(), ensure_ascii=False)))
print("=" * 76)
MARK = {"ok": "[ok]  ", "warn": "[warn]", "fail": "[FAIL]", "info": "[info]", "skip": "[skip]"}
for r in REPORT.rows:
    print("%s %-11s%s  %s" % (MARK.get(r["status"], "[?]   "), r["domain"],
                              r["item"], r["detail"][:120]))

try:
    sh("mkdir -p %s" % shlex.quote(BACKUP_DIR))
    lp = _write_local("bootstrap-report-%s.json" % TS,
                      json.dumps(doc, ensure_ascii=False, indent=2))
    rp = BACKUP_DIR + "/bootstrap-report-%s.json" % TS
    X.put(lp, rp)
    print("-" * 76)
    print("证据落盘: " + rp)
    print("归档: 脱敏后并入 docs/evidence/p0-node-alignment-20260921.md (PRD 第7节)")
except Exception as e:
    print("证据落盘失败: " + repr(e))

gaps = REPORT.gaps()
print("-" * 76)
if gaps:
    print("待处理缺口 %d 项:" % len(gaps))
    for g in gaps:
        print("  [%s] %s/%s -> %s" % (g["status"], g["domain"], g["item"],
                                      g["fix"] or "人工评估"))
else:
    print("无 fail/warn 缺口。")